In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable


In [0]:
(delta_table.alias("t")
 .merge(
     final_df.alias("s"),
     """
     t.job_id = s.job_id
     AND t.workspace_id = s.workspace_id
     AND t.is_current = true
     """
 )
 .whenMatchedUpdate(
     condition="""
     t.name <> s.name OR
     t.description <> s.description OR
     t.paused <> s.paused OR
     t.trigger_type <> s.trigger_type OR
     t.timeout_seconds <> s.timeout_seconds
     """,
     set={
         "effective_end_date": "s.load_timestamp",
         "is_current": "false"
     }
 )
 .whenNotMatchedInsert(values={
     "account_id": "s.account_id",
     "workspace_id": "s.workspace_id",
     "job_id": "s.job_id",
     "name": "s.name",
     "description": "s.description",
     "creator_user_name": "s.creator_user_name",
     "paused": "s.paused",
     "trigger_type": "s.trigger_type",
     "timeout_seconds": "s.timeout_seconds",
     "load_timestamp": "s.load_timestamp",
     "effective_start_date": "s.effective_start_date",
     "effective_end_date": "s.effective_end_date",
     "is_current": "s.is_current"
 })
 .execute()
)